# Results for model: kimi-k2-thinking

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np

df = pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')
df = df.sort('date_id')
dates = df['date_id'].unique().sort()
batch_size = 50
thresholds = []
for i in range(0, len(dates), batch_size):
    batch_dates = dates[i:i+batch_size]
    batch_df = df.filter(pl.col('date_id').is_in(batch_dates))
    thresholds.append(batch_df['feature_00'].quantile(0.85))
threshold = pl.Series(thresholds).quantile(0.85)
df = df.with_columns(((pl.col('feature_00') > threshold).cast(pl.Int32).alias('feature_00_top15')))
features = [c for c in df.columns if c.startswith('feature_')] + ['feature_00_top15']
split_date = dates[int(len(dates) * 0.8)]
X_train = df.filter(pl.col('date_id') < split_date).select(features).to_pandas().fillna(0)
X_val = df.filter(pl.col('date_id') >= split_date).select(features).to_pandas().fillna(0)
y_train = df.filter(pl.col('date_id') < split_date)['responder_6'].to_pandas()
y_val = df.filter(pl.col('date_id') >= split_date)['responder_6'].to_pandas()
model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, objective='reg:squarederror', n_jobs=-1, random_state=42)
model.fit(X_train, y_train)
print(f"Validation R²: {model.score(X_val, y_val):.4f}")